In [8]:
# This test program aims to output a list of n_mc coordinates within a sample.
# This program will use ray-tracing and the odd-even rule to determine whether a random coordinate is within a sample.

using MeshIO
using FileIO
using BenchmarkTools
using StaticArrays
using GeometryBasics
include("crystal.jl")
using .crystal

In [9]:
# Setting the desired number of MC sample points.

const n_mc = 10

10

In [10]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.

# Dummy multiple-crystal sample comprising 7 icospheres with 80 faces each.
stl = load("STL_FileExamples/7_Icospheres80.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

560

In [11]:
# Storing the coordinates into a vector of length n_mc.

mc_coords = Vector{SVector{3, Float32}}(undef, n_mc)

10-element Vector{SVector{3, Float32}}:
 [46.00003, 4.65f-43, 7.0414663f34]
 [4.5914f-41, 7.041419f34, 4.5914f-41]
 [46.019287, 4.65f-43, 7.0606475f34]
 [4.5914f-41, 46.019287, 4.65f-43]
 [7.041419f34, 4.5914f-41, 4.1352142f37]
 [4.5914f-41, 46.00003, 4.65f-43]
 [46.019287, 4.65f-43, 7.0411336f34]
 [4.5914f-41, 2.3163367f35, 4.5914f-41]
 [46.024414, 4.65f-43, 46.026123]
 [4.65f-43, 46.027832, 4.65f-43]

In [12]:
# Calculating and storing the vectors parallel to each face, e2 = V2 - V1 and e3 = V3 - V1, and V1s specifically in preparation for the Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

v1s, e2s, e3s = crystal.ve_calc(vertices, indices)

(SVector{3, Float32}[[-0.58778524, 0.809017, 0.0], [0.0, -1.0, 0.0], [0.58778524, 0.809017, 0.0], [-0.95105654, 0.309017, 0.0], [0.58778524, -0.809017, 0.0], [-0.4253254, -0.309017, 0.8506508], [0.4253254, 0.309017, -0.8506508], [-0.68819094, 0.5, -0.5257311], [-0.5257311, 0.0, -0.8506508], [-0.68819094, -0.5, -0.5257311]  …  [-0.4253254, -0.309017, -2.1493492], [0.16245985, -0.5, -2.1493492], [0.68819094, -0.5, -2.474269], [0.16245985, -0.5, -2.1493492], [0.5257311, 0.0, -2.1493492], [0.68819094, 0.5, -2.474269], [0.5257311, 0.0, -2.1493492], [0.16245985, 0.5, -2.1493492], [-0.26286554, 0.809017, -2.474269], [0.16245985, 0.5, -2.1493492]], SVector{3, Float32}[[0.58778524, 0.190983, 0.0], [-0.58778524, 0.190983, 0.0], [-0.58778524, 0.190983, 0.0], [0.10040575, -0.309017, 0.5257311], [0.10040569, 0.309017, 0.5257311], [-0.4253254, 0.309017, -0.3249197], [-0.16245985, 0.5, 0.3249197], [0.10040569, 0.309017, 0.5257311], [0.36327124, 0.5, 0.0], [0.5257311, 0.0, -0.3249197]  …  [0.16245985,

In [16]:
# Generating the sample points.

# Calculating the extrema of the axis-aligned bounding box around the sample.
ranges = crystal.aabb(vertices)
crystal.sampling!(ranges, e2s, e3s, v1s, mc_coords, n_faces, n_mc)

10-element Vector{SVector{3, Float32}}:
 [-0.5707996, -2.4025648, 0.38813457]
 [-2.405533, 0.37918228, -0.5446985]
 [0.22743404, -0.33039796, 3.114026]
 [3.4346461, -0.14972016, 0.6955924]
 [-0.4599055, 2.420153, 0.36776778]
 [-0.122133195, -3.6514053, -0.13118106]
 [-2.4031904, 0.3448147, -0.15779145]
 [2.8285558, 0.65245765, 0.022869013]
 [0.0560623, 2.7877111, 0.056590393]
 [-0.5792192, 2.6794488, 0.6046275]

In [17]:
# Benchmarking this sample point generation function.

@benchmark crystal.sampling!(ranges, e2s, e3s, v1s, mc_coords, n_faces, n_mc)

BenchmarkTools.Trial: 6855 samples with 1 evaluation per sample.
 Range (min … max):  146.200 μs …  12.687 ms  ┊ GC (min … max): 0.00% … 93.20%
 Time  (median):     679.300 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   721.301 μs ± 294.572 μs  ┊ GC (mean ± σ):  0.24% ±  1.13%

              ▁▃▃▃▆█▇▇█▇▇▅▆▆▅▆▃▃▂▁                               
  ▁▁▁▂▂▂▃▃▅▆▆▇███████████████████████▆▆▅▆▅▄▄▄▄▃▃▃▂▃▂▂▂▂▂▁▂▂▂▂▁▁ ▄
  146 μs           Histogram: frequency by time         1.53 ms <

 Memory estimate: 9.08 KiB, allocs estimate: 10.